In [ ]:
# Cell 00: Setup and dependency check
# Halts if any required package is missing — mirrors 00_env_check.R

required_packages <- c("ggplot2", "RcppTOML", "scales", "patchwork", "dplyr")

missing <- required_packages[
  !sapply(required_packages, requireNamespace, quietly = TRUE)
]

if (length(missing) > 0) {
  stop(paste(
    "HALT: Missing packages:", paste(missing, collapse = ", "),
    "\nInstall with: install.packages(c(",
    paste0('"', missing, '"', collapse = ", "), "))"
  ))
}

suppressPackageStartupMessages({
  library(ggplot2)
  library(RcppTOML)
  library(scales)
  library(patchwork)
  library(dplyr)
})

cat("Runtime check PASS\n")
cat("R version:", R.version$major, ".", R.version$minor, "\n", sep = "")
cat("ggplot2:", as.character(packageVersion("ggplot2")), "\n")
cat("RcppTOML:", as.character(packageVersion("RcppTOML")), "\n")
cat("dplyr:", as.character(packageVersion("dplyr")), "\n")


In [ ]:
# Cell 01: Load configuration
# All scenario parameters sourced from CFG. No hardcoded values.

resolve_toml_path <- function() {
  # Strategy 1: relative from notebooks/ subdirectory (standard layout)
  candidate_nb <- file.path("..", "config", "sira.toml")
  if (file.exists(candidate_nb)) return(normalizePath(candidate_nb))

  # Strategy 2: relative from repo root (launched from root)
  candidate_root <- file.path("config", "sira.toml")
  if (file.exists(candidate_root)) return(normalizePath(candidate_root))

  # Strategy 3: environment variable override (CI / air-gapped deployment)
  env_path <- Sys.getenv("SIRA_TOML_PATH", unset = "")
  if (nchar(env_path) > 0 && file.exists(env_path)) {
    return(normalizePath(env_path))
  }

  # Strategy 4: walk upward from current directory (fallback)
  wd <- getwd()
  for (i in seq_len(5)) {
    candidate_walk <- file.path(wd, "config", "sira.toml")
    if (file.exists(candidate_walk)) return(normalizePath(candidate_walk))
    wd <- dirname(wd)
  }

  stop(paste(
    "HALT: sira.toml not found.\n",
    "Mitigation strategies tried:\n",
    "  1. ../config/sira.toml (from notebooks/ subdirectory)\n",
    "  2. config/sira.toml (from repo root)\n",
    "  3. SIRA_TOML_PATH environment variable\n",
    "  4. Upward directory walk (5 levels)\n",
    "Set SIRA_TOML_PATH=/absolute/path/to/sira.toml to resolve."
  ))
}

TOML_PATH <- resolve_toml_path()
CFG       <- RcppTOML::parseTOML(TOML_PATH)

cat("Configuration loaded from:", TOML_PATH, "\n")
cat("Working directory:        ", getwd(), "\n")
cat("Scenarios declared:       ",
    paste(CFG$scenarios$names, collapse = ", "), "\n")


In [ ]:
# Cell 02: Seed and runtime declaration
# Reproducibility is guaranteed by CFG$runtime$seed.
# Every rnorm() and rbeta() call downstream inherits this seed state.

SEED <- CFG$runtime$seed
N    <- CFG$runtime$n_simulations

set.seed(SEED)

cat("Seed:        ", SEED, "\n")
cat("Simulations: ", N, "\n")
cat("Data mode:   ", CFG$runtime$data_mode, "\n")
cat(
  "Reproducibility: all draws are deterministic from this seed.",
  "Re-run from Cell 02 to reset.\n"
)


In [ ]:
# Cell 03: Scenario parameter display table
# Renders a governance-ready parameter table from CFG.
# This is the declared-intent register — what the model is configured to do.

scenario_names <- CFG$scenarios$names

param_table <- do.call(rbind, lapply(scenario_names, function(s) {
  sc <- CFG$scenarios[[s]]
  data.frame(
    Scenario         = s,
    Distribution     = sc$distribution,
    Shape1           = ifelse(!is.null(sc$shape1),        sc$shape1,        NA),
    Shape2           = ifelse(!is.null(sc$shape2),        sc$shape2,        NA),
    Exponent         = ifelse(!is.null(sc$exponent),      sc$exponent,      NA),
    Ruin_Threshold   = sc$ruin_threshold,
    Shock_Multiplier = ifelse(!is.null(sc$shock_multiplier), sc$shock_multiplier, 1.0),
    FX_Devaluation   = ifelse(!is.null(sc$fx_devaluation),  sc$fx_devaluation,   NA),
    stringsAsFactors = FALSE
  )
}))

# Vol multiplier column — derived from distribution type and shock params
param_table$Vol_Multiplier <- ifelse(
  param_table$Distribution == "beta",
  ifelse(is.na(param_table$Shock_Multiplier), "1.00x",
         paste0(param_table$Shock_Multiplier, "x")),
  ifelse(!is.na(param_table$FX_Devaluation),
         paste0("fx_dev: ", param_table$FX_Devaluation),
         "exponent-implied")
)

print(param_table[, c(
  "Scenario", "Distribution", "Ruin_Threshold",
  "Shock_Multiplier", "Vol_Multiplier"
)])


In [ ]:
# Cell 04: Helper functions
# draw_beta(), draw_powerlaw(), apply_shock(), compute_signals()
# Mirror the logic in scripts/02_analysis.R exactly.
# No scenario values are hardcoded — all parameters passed explicitly.

draw_beta <- function(n, shape1, shape2, recovery_anchor) {
  # Beta draws conditioned on recovery_anchor
  # Bounded support enforced by rbeta — no clipping required at draw
  raw <- rbeta(n, shape1, shape2)
  # Shift toward recovery_anchor (portfolio-level conditioning)
  conditioned <- raw * recovery_anchor + (1 - recovery_anchor) * raw
  # Hard clip to (0.001, 0.995) — economically feasible fraction
  pmin(pmax(conditioned, 0.001), 0.995)
}

draw_powerlaw <- function(n, exponent) {
  # Inverse-transform sampling for Power Law heavy tail
  # P(X > x) = x^(-exponent) for x >= 1
  # Recovery fraction: clip to (0.001, 0.995)
  u <- runif(n)
  raw <- (1 - u)^(-1 / exponent)
  # Normalise to (0,1) recovery fraction
  normalised <- 1 / raw
  pmin(pmax(normalised, 0.001), 0.995)
}

apply_shock <- function(recoveries, shock_multiplier = 1.0,
                        fx_devaluation = NULL) {
  shocked <- recoveries * shock_multiplier
  if (!is.null(fx_devaluation) && !is.na(fx_devaluation)) {
    shocked <- shocked * (1 - fx_devaluation)
  }
  pmin(pmax(shocked, 0.001), 0.995)
}

compute_signals <- function(recoveries, ruin_threshold, zscore_threshold) {
  z    <- scale(recoveries)[, 1]
  ruin <- recoveries <= ruin_threshold
  sell <- ruin | (z <= -abs(zscore_threshold))
  ifelse(sell, "SELL", "HOLD")
}

draw_rnorm_reference <- function(recoveries) {
  # Gaussian reference draw at equivalent mean/sd to SIRA recoveries.
  # Used for structural comparison only — NOT in SIRA signal logic.
  # Deliberate repetition in scenario cells is replaced by this helper.
  # Annotated here so reviewers understand the pattern is governed,
  # not careless duplication.
  #
  # Governance note: Gaussian is structurally inappropriate for
  # distressed recovery (CREDIT_RISK_LAYER.md §3.1). The reference
  # draw quantifies the mis-specification, it does not correct it.
  nr <- rnorm(length(recoveries),
    mean = mean(recoveries),
    sd   = sd(recoveries)
  )
  pmin(pmax(nr, 0.001), 0.995)
}

cat("Helper functions loaded: draw_beta, draw_powerlaw,",
    "apply_shock, compute_signals, draw_rnorm_reference
")


In [ ]:
# Cell 05: Baseline scenario
# Distribution: Beta | Vol multiplier: 1.00x
# rnorm() overlay demonstrates departure from Gaussian — key governance point.
# Gaussian is structurally wrong for distressed recovery (CREDIT_RISK_LAYER.md §3.1)

sc    <- CFG$scenarios[["Baseline"]]
N     <- CFG$runtime$n_simulations
ANCHOR <- CFG$data$recovery_anchor_default

recoveries_baseline <- draw_beta(N, sc$shape1, sc$shape2, ANCHOR)
signals_baseline    <- compute_signals(
  recoveries_baseline, sc$ruin_threshold,
  CFG$signals$sell_zscore_threshold
)

# rnorm draws at equivalent mean/sd — for structural comparison only
# These are NOT used in SIRA signal logic
rnorm_baseline <- draw_rnorm_reference(recoveries_baseline)

df_baseline <- data.frame(
  recovery = c(recoveries_baseline, rnorm_baseline),
  source   = rep(c("Beta (SIRA)", "Normal (reference)"), each = N),
  signal   = c(signals_baseline, rep(NA, N))
)

p_baseline <- ggplot(df_baseline, aes(x = recovery, fill = source)) +
  geom_density(alpha = 0.55, colour = NA) +
  geom_vline(
    xintercept = sc$ruin_threshold,
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.7
  ) +
  annotate("text",
    x = sc$ruin_threshold + 0.02, y = Inf,
    label = paste("Ruin threshold:", sc$ruin_threshold),
    hjust = 0, vjust = 1.4, size = 3,
    colour = "#E24B4A"
  ) +
  scale_fill_manual(
    values = c("Beta (SIRA)" = "#185FA5", "Normal (reference)" = "#888780")
  ) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  labs(
    title    = "Baseline — Beta recovery distribution vs Normal reference",
    subtitle = paste0(
      "shape1=", sc$shape1, "  shape2=", sc$shape2,
      "  ruin=", sc$ruin_threshold,
      "  n=", scales::comma(N),
      "  seed=", CFG$runtime$seed
    ),
    x    = "Stressed recovery",
    y    = "Density",
    fill = NULL,
    caption = "Normal reference shown for structural comparison only — not used in SIRA signal logic."
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

print(p_baseline)

cat("\nBaseline signal summary:\n")
print(table(signals_baseline))


In [ ]:
# Cell 06: Liquidity Crunch scenario
# Distribution: Beta | Vol multiplier: shock_multiplier
# Shock compresses recoveries toward lower bound.
# rnorm overlay shows Gaussian underestimates lower-tail mass.

sc <- CFG$scenarios[["Liquidity_Crunch"]]

recoveries_raw  <- draw_beta(N, sc$shape1, sc$shape2, ANCHOR)
recoveries_lc   <- apply_shock(recoveries_raw, sc$shock_multiplier)
signals_lc      <- compute_signals(
  recoveries_lc, sc$ruin_threshold,
  CFG$signals$sell_zscore_threshold
)

rnorm_lc <- draw_rnorm_reference(recoveries_lc)

df_lc <- data.frame(
  recovery = c(recoveries_lc, rnorm_lc),
  source   = rep(c("Beta shocked (SIRA)", "Normal (reference)"), each = N)
)

p_lc <- ggplot(df_lc, aes(x = recovery, fill = source)) +
  geom_density(alpha = 0.55, colour = NA) +
  geom_vline(
    xintercept = sc$ruin_threshold,
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.7
  ) +
  annotate("text",
    x = sc$ruin_threshold + 0.02, y = Inf,
    label = paste("Ruin:", sc$ruin_threshold),
    hjust = 0, vjust = 1.4, size = 3, colour = "#E24B4A"
  ) +
  scale_fill_manual(
    values = c(
      "Beta shocked (SIRA)" = "#0F6E56",
      "Normal (reference)"  = "#888780"
    )
  ) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  labs(
    title    = "Liquidity Crunch — Beta with shock multiplier",
    subtitle = paste0(
      "shape1=", sc$shape1, "  shape2=", sc$shape2,
      "  shock=", sc$shock_multiplier,
      "  ruin=", sc$ruin_threshold
    ),
    x = "Stressed recovery", y = "Density", fill = NULL,
    caption = paste0(
      "Shock multiplier ", sc$shock_multiplier,
      "x applied post-draw. Vol multiplier: ", sc$shock_multiplier, "x."
    )
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

print(p_lc)

cat("\nLiquidity Crunch signal summary:\n")
print(table(signals_lc))


In [ ]:
# Cell 07: Jurisdictional Freeze scenario
# Distribution: Beta | Vol multiplier: shock_multiplier
# Recovery collapses toward ruin threshold.
# Chart highlights the proximity collapse mechanism.

sc <- CFG$scenarios[["Jurisdictional_Freeze"]]

recoveries_raw <- draw_beta(N, sc$shape1, sc$shape2, ANCHOR)
recoveries_jf  <- apply_shock(recoveries_raw, sc$shock_multiplier)
signals_jf     <- compute_signals(
  recoveries_jf, sc$ruin_threshold,
  CFG$signals$sell_zscore_threshold
)

rnorm_jf <- draw_rnorm_reference(recoveries_jf)

# Proximity to ruin — key distributional property for this scenario
proximity <- recoveries_jf - sc$ruin_threshold

df_jf <- data.frame(
  recovery  = recoveries_jf,
  proximity = proximity,
  signal    = signals_jf
)

p_jf_dist <- ggplot(df_jf, aes(x = recovery, fill = signal)) +
  geom_histogram(bins = 60, alpha = 0.8, colour = NA) +
  geom_vline(
    xintercept = sc$ruin_threshold,
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.8
  ) +
  scale_fill_manual(values = c("SELL" = "#E24B4A", "HOLD" = "#185FA5")) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  labs(
    title    = "Jurisdictional Freeze — recovery vs ruin threshold",
    subtitle = paste0(
      "shape1=", sc$shape1, "  shape2=", sc$shape2,
      "  shock=", sc$shock_multiplier,
      "  ruin=", sc$ruin_threshold
    ),
    x = "Stressed recovery", y = "Count", fill = "Signal"
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

p_jf_prox <- ggplot(df_jf, aes(x = proximity, fill = signal)) +
  geom_density(alpha = 0.65, colour = NA) +
  geom_vline(xintercept = 0,
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.7) +
  scale_fill_manual(values = c("SELL" = "#E24B4A", "HOLD" = "#185FA5")) +
  scale_x_continuous(labels = percent_format(accuracy = 1)) +
  labs(
    title    = "Proximity to ruin threshold",
    subtitle = "recovery − ruin_threshold  |  negative = ruin event",
    x = "Proximity", y = "Density", fill = "Signal"
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

print(p_jf_dist / p_jf_prox)

cat("\nJurisdictional Freeze signal summary:\n")
print(table(signals_jf))


In [ ]:
# Cell 08: Counterparty Default scenario
# Distribution: Power Law | Vol multiplier: exponent-implied
# Heavy tail — extreme low-recovery events materially more probable.
# rnorm overlay quantifies the tail mis-specification of Gaussian.

sc <- CFG$scenarios[["Counterparty_Default"]]

recoveries_cd <- draw_powerlaw(N, sc$exponent)
recoveries_cd <- apply_shock(recoveries_cd, sc$shock_multiplier)
signals_cd    <- compute_signals(
  recoveries_cd, sc$ruin_threshold,
  CFG$signals$sell_zscore_threshold
)

rnorm_cd <- draw_rnorm_reference(recoveries_cd)

df_cd <- data.frame(
  recovery = c(recoveries_cd, rnorm_cd),
  source   = rep(c("Power Law (SIRA)", "Normal (reference)"), each = N)
)

# Log-scale density to surface tail behaviour
p_cd <- ggplot(df_cd, aes(x = recovery, colour = source)) +
  geom_density(linewidth = 0.9, fill = NA) +
  geom_vline(
    xintercept = sc$ruin_threshold,
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.7
  ) +
  scale_colour_manual(
    values = c(
      "Power Law (SIRA)"   = "#993C1D",
      "Normal (reference)" = "#888780"
    )
  ) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  scale_y_log10() +
  labs(
    title    = "Counterparty Default — Power Law vs Normal (log-scale density)",
    subtitle = paste0(
      "exponent=", sc$exponent,
      "  shock=", sc$shock_multiplier,
      "  ruin=", sc$ruin_threshold,
      "  vol_multiplier=exponent-implied"
    ),
    x = "Stressed recovery", y = "Density (log scale)", colour = NULL,
    caption = paste0(
      "Log scale surfaces Power Law lower-tail excess over Normal. ",
      "Gaussian underestimates ruin-zone mass by construction."
    )
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

print(p_cd)

cat("\nCounterparty Default signal summary:\n")
print(table(signals_cd))


In [ ]:
# Cell 09: Hyper-Inflationary scenario
# Distribution: Power Law | Vol multiplier: fx_devaluation
# FX devaluation impairs real bond value post-draw.
# Chart shows pre- and post-devaluation distributions side by side.

sc <- CFG$scenarios[["Hyper_Inflationary"]]

recoveries_pre  <- draw_powerlaw(N, sc$exponent)
recoveries_pre  <- apply_shock(recoveries_pre, sc$shock_multiplier)
recoveries_post <- apply_shock(
  recoveries_pre,
  fx_devaluation = sc$fx_devaluation
)
signals_hi      <- compute_signals(
  recoveries_post, sc$ruin_threshold,
  CFG$signals$sell_zscore_threshold
)

rnorm_hi <- draw_rnorm_reference(recoveries_post)

df_hi <- data.frame(
  recovery = c(recoveries_pre, recoveries_post, rnorm_hi),
  source   = rep(c(
    "Power Law pre-FX",
    "Power Law post-FX (SIRA)",
    "Normal (reference)"
  ), each = N)
)

p_hi <- ggplot(df_hi, aes(x = recovery, fill = source)) +
  geom_density(alpha = 0.50, colour = NA) +
  geom_vline(
    xintercept = sc$ruin_threshold,
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.7
  ) +
  annotate("text",
    x = sc$ruin_threshold + 0.02, y = Inf,
    label = paste("Ruin:", sc$ruin_threshold),
    hjust = 0, vjust = 1.4, size = 3, colour = "#E24B4A"
  ) +
  scale_fill_manual(
    values = c(
      "Power Law pre-FX"         = "#BA7517",
      "Power Law post-FX (SIRA)" = "#993C1D",
      "Normal (reference)"       = "#888780"
    )
  ) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  labs(
    title    = "Hyper-Inflationary — Power Law with FX devaluation impairment",
    subtitle = paste0(
      "exponent=", sc$exponent,
      "  fx_devaluation=", sc$fx_devaluation,
      "  shock=", sc$shock_multiplier,
      "  ruin=", sc$ruin_threshold
    ),
    x = "Stressed recovery", y = "Density", fill = NULL,
    caption = paste0(
      "FX devaluation=", sc$fx_devaluation,
      " applied to real bond value post Power Law draw. ",
      "Vol multiplier: fx_devaluation."
    )
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

print(p_hi)

cat("\nHyper-Inflationary signal summary:\n")
print(table(signals_hi))


In [ ]:
# Cell 10: Cross-scenario comparison
# Faceted density chart — all five scenarios in a single governed view.
# rnorm reference overlay on every facet quantifies Gaussian mis-specification
# per scenario. This is the committee-presentation chart.

scenarios_all <- list(
  Baseline             = list(r = recoveries_baseline, s = signals_baseline),
  Liquidity_Crunch     = list(r = recoveries_lc,       s = signals_lc),
  Jurisdictional_Freeze= list(r = recoveries_jf,       s = signals_jf),
  Counterparty_Default = list(r = recoveries_cd,       s = signals_cd),
  Hyper_Inflationary   = list(r = recoveries_post,     s = signals_hi)
)

df_all <- do.call(rbind, lapply(names(scenarios_all), function(nm) {
  sc_cfg <- CFG$scenarios[[nm]]
  r      <- scenarios_all[[nm]]$r
  s      <- scenarios_all[[nm]]$s
  nr     <- draw_rnorm_reference(r)

  rbind(
    data.frame(
      scenario     = nm,
      recovery     = r,
      source       = "SIRA",
      signal       = s,
      ruin         = sc_cfg$ruin_threshold,
      distribution = sc_cfg$distribution,
      stringsAsFactors = FALSE
    ),
    data.frame(
      scenario     = nm,
      recovery     = nr,
      source       = "Normal ref",
      signal       = NA_character_,
      ruin         = sc_cfg$ruin_threshold,
      distribution = "normal",
      stringsAsFactors = FALSE
    )
  )
}))

# Scenario display labels with distribution and vol multiplier
scenario_labels <- c(
  Baseline              = "Baseline\nBeta | 1.00x",
  Liquidity_Crunch      = "Liquidity Crunch\nBeta | shock_multiplier",
  Jurisdictional_Freeze = "Jurisdictional Freeze\nBeta | shock_multiplier",
  Counterparty_Default  = "Counterparty Default\nPower Law | exponent-implied",
  Hyper_Inflationary    = "Hyper-Inflationary\nPower Law | fx_devaluation"
)
df_all$scenario_label <- scenario_labels[df_all$scenario]
df_all$scenario_label <- factor(
  df_all$scenario_label,
  levels = scenario_labels
)

# Ruin threshold per facet
df_ruin <- unique(df_all[, c("scenario_label", "ruin")])

p_cross <- ggplot(
  df_all[df_all$source == "SIRA", ],
  aes(x = recovery, fill = signal)
) +
  geom_density(alpha = 0.70, colour = NA) +
  geom_density(
    data = df_all[df_all$source == "Normal ref", ],
    aes(x = recovery),
    inherit.aes = FALSE,
    fill = "#888780", alpha = 0.25, colour = NA
  ) +
  geom_vline(
    data = df_ruin,
    aes(xintercept = ruin),
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.6
  ) +
  facet_wrap(~ scenario_label, ncol = 1, scales = "free_y") +
  scale_fill_manual(
    values = c("SELL" = "#E24B4A", "HOLD" = "#185FA5"),
    na.value = "#888780"
  ) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  labs(
    title   = "SIRA — Five stress scenarios: recovery distributions",
    subtitle = paste0(
      "Seed: ", CFG$runtime$seed,
      "  |  n=", scales::comma(N),
      "  |  Grey: Normal reference  |  Dashed: ruin threshold"
    ),
    x    = "Stressed recovery",
    y    = "Density",
    fill = "Signal",
    caption = paste0(
      "SIRA stochastic recovery engine. ",
      "Beta scenarios: Baseline, Liquidity Crunch, Jurisdictional Freeze. ",
      "Power Law scenarios: Counterparty Default, Hyper-Inflationary. ",
      "Normal reference shown for structural comparison only."
    )
  ) +
  theme_minimal(base_size = 11) +
  theme(
    legend.position  = "bottom",
    strip.text       = element_text(size = 9, face = "plain"),
    panel.spacing    = unit(1.2, "lines")
  )

print(p_cross)


In [ ]:
# Cell 11: SELL/HOLD signal summary table
# Governance-ready summary. Matches terminal emission format from
# scripts/03_visualize.R. Suitable for committee review.

summary_rows <- lapply(names(scenarios_all), function(nm) {
  sc_cfg  <- CFG$scenarios[[nm]]
  r       <- scenarios_all[[nm]]$r
  s       <- scenarios_all[[nm]]$s
  n_sell  <- sum(s == "SELL")
  n_hold  <- sum(s == "HOLD")
  n_total <- length(s)

  data.frame(
    Scenario        = nm,
    Distribution    = sc_cfg$distribution,
    Vol_Multiplier  = ifelse(
      sc_cfg$distribution == "beta",
      ifelse(!is.null(sc_cfg$shock_multiplier),
             paste0(sc_cfg$shock_multiplier, "x"), "1.00x"),
      ifelse(!is.null(sc_cfg$fx_devaluation),
             paste0("fx_dev: ", sc_cfg$fx_devaluation),
             "exponent-implied")
    ),
    Ruin_Threshold  = sc_cfg$ruin_threshold,
    Mean_Recovery   = round(mean(r), 4),
    SD_Recovery     = round(sd(r),   4),
    N_SELL          = n_sell,
    N_HOLD          = n_hold,
    SELL_pct        = paste0(round(100 * n_sell / n_total, 1), "%"),
    stringsAsFactors = FALSE
  )
})

summary_table <- do.call(rbind, summary_rows)

cat("\n")
cat(strrep("─", 80), "\n")
cat("SIRA — Five Scenario Signal Summary\n")
cat(strrep("─", 80), "\n")
print(summary_table, row.names = FALSE)
cat(strrep("─", 80), "\n")

# Worst-case scenario by SELL count
worst <- summary_table[which.max(summary_table$N_SELL), "Scenario"]
cat("Worst-case scenario:", worst, "\n")
cat(strrep("─", 80), "\n")


In [ ]:
# Cell 12: Session metadata and reproducibility declaration
# This cell produces the evidence record for the notebook run.
# Suitable for inclusion in compliance evidence packs.

cat(strrep("═", 80), "\n")
cat("SIRA NOTEBOOK — SESSION METADATA\n")
cat(strrep("═", 80), "\n")
cat("Timestamp (UTC): ", format(Sys.time(), tz = "UTC", usetz = TRUE), "\n")
cat("Seed:            ", CFG$runtime$seed, "\n")
cat("Simulations:     ", scales::comma(N), "\n")
cat("Config source:   ", normalizePath(TOML_PATH), "\n")
cat("R version:       ",
    R.version$major, ".", R.version$minor, "\n", sep = "")
cat("ggplot2:         ", as.character(packageVersion("ggplot2")), "\n")
cat("RcppTOML:        ", as.character(packageVersion("RcppTOML")), "\n")
cat("Scenarios run:   ", paste(names(scenarios_all), collapse = ", "), "\n")
cat(strrep("─", 80), "\n")
cat("Reproducibility: all stochastic draws are deterministic from the\n")
cat("declared seed (", CFG$runtime$seed, "). Re-run from Cell 02 to reset.\n",
    sep = "")
cat("This notebook does not write to output/ or modify any governed\n")
cat("artefact. It is a read-only inspection surface over the SIRA engine.\n")
cat(strrep("═", 80), "\n")


In [ ]:
# Cell 13: Capital stack — liability engine
# Mirrors scripts/10_liability_engine.R
# Consumes CFG$liability. No hardcoded values.
# Outputs: annual_obligation, PV of obligations, solvency_headroom
# Unit: NZD (or operator-declared currency in CFG)

pool   <- CFG$liability$annuity_pool_size
rate   <- CFG$liability$payout_rate
dur    <- CFG$liability$duration_years
infl   <- CFG$liability$inflation_sensitivity
buffer <- CFG$liability$insurer_solvency_buffer

annual_obligation <- pool * rate

# Present value of obligation stream (flat coupon, simple discounting)
# PV = annual_obligation * sum(1/(1+r)^t) for t = 1..duration
discount_rate <- CFG$credit$gross_yield  # use credit yield as discount proxy
pv_obligations <- annual_obligation * sum(
  1 / (1 + discount_rate)^(seq_len(dur))
)

solvency_headroom <- pool * (1 - buffer) - pv_obligations

cat(strrep("─", 60), "
")
cat("Liability Engine
")
cat(strrep("─", 60), "
")
cat("Annuity pool:        ", scales::dollar(pool), "
")
cat("Annual obligation:   ", scales::dollar(annual_obligation), "
")
cat("PV obligations:      ", scales::dollar(pv_obligations), "
")
cat("Solvency headroom:   ", scales::dollar(solvency_headroom), "
")
cat("Headroom signal:     ",
    ifelse(solvency_headroom > 0, "SOLVENT", "BREACH"), "
")
cat(strrep("─", 60), "
")

# Hyper-Inflationary stress on fixed payout
# Fixed payouts lose real value — real obligation rises
real_obligation_stressed <- annual_obligation *
  (1 + CFG$scenarios$Hyper_Inflationary$fx_devaluation * infl)
cat("Stressed obligation (Hyper-Inflationary real):",
    scales::dollar(real_obligation_stressed), "
")

ANNUAL_OBLIGATION <- annual_obligation   # pass forward to Cell 14
SOLVENCY_HEADROOM <- solvency_headroom


In [ ]:
# Cell 14: Capital stack — credit deployment + spread stress aggregator
# Mirrors scripts/11_credit_deployment.R and scripts/12_spread_stress.R
# Consumes scenario-level net income from stochastic recovery layer.
# Spread signal: SOLVENT / WATCH / BREACH per scenario.

deployed    <- CFG$credit$deployed_capital
gross_yield <- CFG$credit$gross_yield
fee_income  <- CFG$credit$fee_income
kicker      <- CFG$credit$equity_kicker_value
default_rate<- CFG$credit$default_rate_assumption

gross_income <- deployed * gross_yield + fee_income + kicker

# Scenario-level net income: apply scenario stressed_recovery as LGD proxy
# LGD_proxy = 1 - stressed_recovery (from CREDIT_RISK_LAYER.md §1.1)
spread_results <- lapply(names(scenarios_all), function(nm) {
  r         <- scenarios_all[[nm]]$r
  mean_rec  <- mean(r)
  lgd_proxy <- 1 - mean_rec
  expected_loss <- deployed * lgd_proxy * default_rate
  net_income    <- gross_income - expected_loss
  spread        <- net_income - ANNUAL_OBLIGATION

  signal <- dplyr::case_when(
    spread > 0 & SOLVENCY_HEADROOM > 0 ~ "SOLVENT",
    spread > 0 & SOLVENCY_HEADROOM <= 0 ~ "WATCH",
    TRUE ~ "BREACH"
  )

  data.frame(
    Scenario      = nm,
    Mean_Recovery = round(mean_rec, 4),
    LGD_Proxy     = round(lgd_proxy, 4),
    Net_Income    = round(net_income, 0),
    Spread        = round(spread, 0),
    Signal        = signal,
    stringsAsFactors = FALSE
  )
})

spread_table <- do.call(rbind, spread_results)

cat(strrep("─", 70), "
")
cat("Capital Stack — Spread Stress Summary
")
cat(strrep("─", 70), "
")
print(spread_table, row.names = FALSE)
cat(strrep("─", 70), "
")

# Pass net income forward to valuation layer
NET_INCOME_BY_SCENARIO <- setNames(
  sapply(spread_results, function(x) x$Net_Income),
  names(scenarios_all)
)


In [ ]:
# Cell 15: Capital stack — spread visualisation
# Grouped bar: spread captured vs obligation rate per scenario.
# Colour: SOLVENT=teal, WATCH=amber, BREACH=red.

signal_colours <- c(
  "SOLVENT" = "#1D9E75",
  "WATCH"   = "#BA7517",
  "BREACH"  = "#E24B4A"
)

p_spread <- ggplot(spread_table,
    aes(x = reorder(Scenario, Spread), y = Spread / 1e6,
        fill = Signal)) +
  geom_col(alpha = 0.85, width = 0.6) +
  geom_hline(yintercept = 0,
    colour = "#444441", linewidth = 0.6, linetype = "solid") +
  coord_flip() +
  scale_fill_manual(values = signal_colours) +
  scale_y_continuous(labels = scales::label_dollar(suffix = "M")) +
  labs(
    title    = "Capital Stack — Net spread by scenario",
    subtitle = paste0(
      "Gross yield: ", scales::percent(CFG$credit$gross_yield),
      "  |  Obligation rate: ", scales::percent(CFG$liability$payout_rate),
      "  |  Deployed: ", scales::dollar(CFG$credit$deployed_capital)
    ),
    x = NULL, y = "Net spread (M)", fill = "Signal",
    caption = "Negative spread = annuity obligations exceed net credit income under scenario stress."
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

print(p_spread)


In [ ]:
# Cell 16: Valuation — stress-conditioned DCF
# Mirrors scripts/20_dcf.R
# Formula: PV_stressed = sum(coupon/(1+r_stressed)^t)
#          + (stressed_recovery * face_value) / (1+r_stressed)^T
# Units: currency (CFG-declared)

face      <- CFG$dcf$face_value
coupon    <- face * CFG$dcf$coupon_rate
r_base    <- CFG$dcf$discount_rate_base
r_stressed<- CFG$dcf$discount_rate_stressed
term      <- CFG$dcf$term_years
threshold <- CFG$dcf$impairment_threshold

anchor_default <- ifelse(
  !is.null(CFG$data$recovery_anchor_default),
  CFG$data$recovery_anchor_default,
  CFG$dcf$recovery_override
)

dcf_base <- sum(coupon / (1 + r_base)^seq_len(term)) +
            (face * anchor_default) /
            (1 + r_base)^term

dcf_results <- lapply(names(scenarios_all), function(nm) {
  mean_rec   <- mean(scenarios_all[[nm]]$r)
  pv_coupons <- sum(coupon / (1 + r_stressed)^seq_len(term))
  pv_terminal<- (face * mean_rec) / (1 + r_stressed)^term
  pv_stressed <- pv_coupons + pv_terminal
  erosion     <- (dcf_base - pv_stressed) / dcf_base

  data.frame(
    Scenario      = nm,
    DCF_Base      = round(dcf_base, 0),
    DCF_Stressed  = round(pv_stressed, 0),
    Erosion_pct   = round(100 * erosion, 2),
    Signal        = ifelse(erosion > threshold, "IMPAIRED", "STABLE"),
    stringsAsFactors = FALSE
  )
})

dcf_table <- do.call(rbind, dcf_results)

cat(strrep("─", 60), "
")
cat("DCF Stress — Impairment Summary
")
cat(strrep("─", 60), "
")
print(dcf_table, row.names = FALSE)
cat(strrep("─", 60), "
")

DCF_BY_SCENARIO <- setNames(
  sapply(dcf_results, function(x) x$DCF_Stressed),
  names(scenarios_all)
)


In [ ]:
# Cell 17: Valuation — LBO viability and IRR attribution
# Mirrors scripts/23_lbo.R and scripts/24_irr.R
# DSCR = ebitda_proxy / (debt_amount * debt_cost)
# Attribution: coupon, fee, kicker, recovery contributions

equity  <- CFG$lbo$equity_contribution
debt    <- CFG$lbo$debt_amount
cost    <- CFG$lbo$debt_cost
hold    <- CFG$lbo$hold_period_years
ebitda  <- CFG$lbo$ebitda_proxy
min_dscr<- CFG$lbo$min_dscr
min_irr <- CFG$lbo$min_irr_threshold

dscr           <- ebitda / (debt * cost)
entry_leverage <- debt / (equity + debt)

lbo_irr_results <- lapply(names(scenarios_all), function(nm) {
  # Exit value under stress = DCF stressed value as proxy
  exit_ev       <- DCF_BY_SCENARIO[[nm]]
  equity_return <- exit_ev - debt
  irr_stressed  <- (equity_return / equity)^(1 / hold) - 1

  initial_inv <- equity + debt
  coupons_pv  <- sum(
    (initial_inv * CFG$dcf$coupon_rate) /
    (1 + irr_stressed)^seq_len(hold)
  )
  coupon_contrib   <- coupons_pv / initial_inv
  fee_contrib      <- CFG$credit$fee_income / initial_inv
  kicker_contrib   <- CFG$credit$equity_kicker_value / initial_inv
  recovery_contrib <- (exit_ev - debt) / initial_inv

  signal <- dplyr::case_when(
    dscr >= min_dscr & irr_stressed >= min_irr ~ "VIABLE",
    dscr >= min_dscr & irr_stressed < min_irr  ~ "MARGINAL",
    TRUE                                        ~ "IMPAIRED"
  )

  data.frame(
    Scenario         = nm,
    DSCR             = round(dscr, 3),
    IRR_Stressed_pct = round(100 * irr_stressed, 2),
    Coupon_Contrib   = round(coupon_contrib, 3),
    Fee_Contrib      = round(fee_contrib, 3),
    Kicker_Contrib   = round(kicker_contrib, 3),
    Recovery_Contrib = round(recovery_contrib, 3),
    LBO_Signal       = signal,
    stringsAsFactors = FALSE
  )
})

lbo_table <- do.call(rbind, lbo_irr_results)

cat(strrep("─", 70), "
")
cat("LBO / IRR Stress — Viability and Attribution
")
cat(strrep("─", 70), "
")
cat("DSCR (constant across scenarios):", round(dscr, 3), "
")
cat("Entry leverage:", scales::percent(entry_leverage), "
")
cat(strrep("─", 70), "
")
print(lbo_table, row.names = FALSE)
cat(strrep("─", 70), "
")


In [ ]:
# Cell 18: Black-Scholes options layer
# Mirrors scripts/30_bs_core.R, scripts/30b_greeks.R,
#          scripts/34_vol_surface.R, scripts/31_recovery_put.R
# Derivation basis: Haugh (2016), Columbia IEOR E4706

bsm_d1 <- function(S, K, r, sigma, T) {
  (log(S / K) + (r + 0.5 * sigma^2) * T) / (sigma * sqrt(T))
}
bsm_d2 <- function(d1, sigma, T) d1 - sigma * sqrt(T)

bsm_call <- function(S, K, r, sigma, T, q = 0) {
  d1 <- bsm_d1(S, K, r, sigma, T)
  d2 <- bsm_d2(d1, sigma, T)
  exp(-q * T) * S * pnorm(d1) - exp(-r * T) * K * pnorm(d2)
}

bsm_put <- function(S, K, r, sigma, T, q = 0) {
  call <- bsm_call(S, K, r, sigma, T, q)
  call - exp(-q * T) * S + exp(-r * T) * K
}

greek_delta_put <- function(S, K, r, sigma, T, q = 0) {
  d1 <- bsm_d1(S, K, r, sigma, T)
  exp(-q * T) * pnorm(d1) - exp(-q * T)
}
greek_vega <- function(S, K, r, sigma, T, q = 0) {
  d1 <- bsm_d1(S, K, r, sigma, T)
  exp(-q * T) * S * sqrt(T) * dnorm(d1)
}
greek_gamma <- function(S, K, r, sigma, T, q = 0) {
  d1 <- bsm_d1(S, K, r, sigma, T)
  exp(-q * T) * dnorm(d1) / (S * sigma * sqrt(T))
}

base_sigma <- CFG$vol_surface$base_firm_vol
r_free     <- CFG$options$recovery_put$risk_free_rate
S_firm     <- CFG$options$recovery_put$firm_value
K_debt     <- CFG$options$recovery_put$debt_face_value
T_horizon  <- CFG$options$recovery_put$resolution_horizon

vol_multipliers <- list(
  Baseline              = 1.0,
  Liquidity_Crunch      = CFG$scenarios$Liquidity_Crunch$shock_multiplier,
  Jurisdictional_Freeze = CFG$scenarios$Jurisdictional_Freeze$shock_multiplier,
  Counterparty_Default  = NULL,
  Hyper_Inflationary    = 1 + CFG$scenarios$Hyper_Inflationary$fx_devaluation
)

p_ruin_cd <- mean(signals_cd == "SELL")
sigma_cd_implied <- if (p_ruin_cd > 0 && p_ruin_cd < 1) {
  sqrt(-2 * log(p_ruin_cd) / T_horizon)
} else {
  base_sigma * 2.0
}
vol_multipliers$Counterparty_Default <- sigma_cd_implied / base_sigma

bsm_results <- lapply(names(scenarios_all), function(nm) {
  mult     <- vol_multipliers[[nm]]
  sigma_sc <- base_sigma * mult

  mean_rec    <- mean(scenarios_all[[nm]]$r)
  V_stressed  <- S_firm * mean_rec
  E_stressed  <- max(V_stressed - K_debt, 0.001)
  sigma_equity_stressed <- (V_stressed / E_stressed) * sigma_sc

  put_base     <- bsm_put(S_firm,     K_debt, r_free, base_sigma, T_horizon)
  put_stressed <- bsm_put(V_stressed, K_debt, r_free, sigma_sc,   T_horizon)

  implied_rec_bsm <- max(V_stressed - K_debt, 0) / K_debt
  convergence     <- abs(implied_rec_bsm - mean_rec) <=
                     CFG$options$convergence_tolerance

  delta <- greek_delta_put(V_stressed, K_debt, r_free, sigma_sc, T_horizon)
  vega  <- greek_vega(     V_stressed, K_debt, r_free, sigma_sc, T_horizon)
  gamma <- greek_gamma(    V_stressed, K_debt, r_free, sigma_sc, T_horizon)

  delta_S  <- V_stressed - S_firm
  delta_sig<- sigma_sc - base_sigma
  pnl_delta <- delta * delta_S
  pnl_gamma <- 0.5 * gamma * delta_S^2
  pnl_vega  <- vega * delta_sig

  data.frame(
    Scenario          = nm,
    Vol_Multiplier    = round(mult, 3),
    Sigma_Stressed    = round(sigma_sc, 4),
    Sigma_Equity      = round(sigma_equity_stressed, 4),
    Put_Base          = round(put_base, 2),
    Put_Stressed      = round(put_stressed, 2),
    Implied_Rec_BSM   = round(implied_rec_bsm, 4),
    SIRA_Rec          = round(mean_rec, 4),
    Convergence       = ifelse(convergence, "CONVERGENT", "DIVERGENT"),
    Delta             = round(delta, 4),
    Vega              = round(vega, 4),
    PnL_Delta         = round(pnl_delta, 2),
    PnL_Gamma         = round(pnl_gamma, 2),
    PnL_Vega          = round(pnl_vega, 2),
    stringsAsFactors  = FALSE
  )
})

bsm_table <- do.call(rbind, bsm_results)

cat(strrep("─", 70), "
")
cat("BSM Recovery Put — Scenario Vol Surface and Greeks
")
cat(strrep("─", 70), "
")
cat("[BSM NOTICE] Vol surface is scenario-governed, not market-calibrated.
")
cat("Merton linkage is explanatory framing. Firm value V is operator-declared.
")
cat(strrep("─", 70), "
")
print(bsm_table[, c(
  "Scenario", "Vol_Multiplier", "Put_Stressed",
  "Convergence", "Delta", "Vega"
)], row.names = FALSE)
cat(strrep("─", 70), "
")


In [ ]:
# Cell 19: BSM visualisation
# Panel 1: Put price base vs stressed per scenario
# Panel 2: Delta and Vega surface across scenarios

df_bsm_long <- rbind(
  data.frame(
    Scenario = bsm_table$Scenario,
    Value    = bsm_table$Put_Base,
    Series   = "Put base",
    stringsAsFactors = FALSE
  ),
  data.frame(
    Scenario = bsm_table$Scenario,
    Value    = bsm_table$Put_Stressed,
    Series   = "Put stressed",
    stringsAsFactors = FALSE
  )
)

p_put <- ggplot(df_bsm_long,
    aes(x = reorder(Scenario, Value), y = Value, fill = Series)) +
  geom_col(position = "dodge", alpha = 0.85, width = 0.6) +
  coord_flip() +
  scale_fill_manual(
    values = c("Put base" = "#888780", "Put stressed" = "#993C1D")
  ) +
  scale_y_continuous(labels = scales::dollar_format()) +
  labs(
    title    = "BSM Recovery Put — Base vs stressed put price",
    subtitle = "Firm value V is operator-declared. Vol surface is scenario-governed.",
    x = NULL, y = "Put price", fill = NULL
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

df_greeks <- data.frame(
  Scenario = bsm_table$Scenario,
  Delta    = bsm_table$Delta,
  Vega     = bsm_table$Vega,
  stringsAsFactors = FALSE
)

p_greeks <- ggplot(df_greeks) +
  geom_segment(
    aes(x = reorder(Scenario, Delta),
        xend = reorder(Scenario, Delta),
        y = 0, yend = Delta, colour = "Delta"),
    linewidth = 1.2
  ) +
  geom_point(
    aes(x = reorder(Scenario, Delta), y = Delta, colour = "Delta"),
    size = 3
  ) +
  geom_point(
    aes(x = reorder(Scenario, Delta), y = Vega / max(abs(Vega)),
        colour = "Vega (normalised)"),
    size = 2.5, shape = 17
  ) +
  coord_flip() +
  scale_colour_manual(
    values = c("Delta" = "#185FA5", "Vega (normalised)" = "#BA7517")
  ) +
  labs(
    title    = "Greeks surface — Delta and Vega by scenario",
    subtitle = "Vega normalised to [-1,1] for dual-axis display. Raw values in BSM table.",
    x = NULL, y = "Greek value", colour = NULL
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

print(p_put / p_greeks)


In [ ]:
# Cell 20: Full architecture deal summary
# Unified committee table consuming all four layers:
#   Core engine (SELL/HOLD), Capital stack (SOLVENT/WATCH/BREACH),
#   Valuation (IMPAIRED/STABLE, LBO signal), BSM (convergence)

deal_summary <- do.call(rbind, lapply(names(scenarios_all), function(nm) {
  core_sell_pct <- round(
    100 * sum(scenarios_all[[nm]]$s == "SELL") / N, 1
  )
  spread_sig <- spread_table[spread_table$Scenario == nm, "Signal"]
  dcf_sig    <- dcf_table[dcf_table$Scenario == nm, "Signal"]
  lbo_sig    <- lbo_table[lbo_table$Scenario == nm, "LBO_Signal"]
  bsm_conv   <- bsm_table[bsm_table$Scenario == nm, "Convergence"]
  put_stress <- bsm_table[bsm_table$Scenario == nm, "Put_Stressed"]

  data.frame(
    Scenario      = nm,
    SELL_pct      = paste0(core_sell_pct, "%"),
    Spread_Signal = spread_sig,
    DCF_Signal    = dcf_sig,
    LBO_Signal    = lbo_sig,
    BSM_Conv      = bsm_conv,
    Put_Stressed  = scales::dollar(put_stress),
    stringsAsFactors = FALSE
  )
}))

cat(strrep("═", 80), "
")
cat("SIRA — FULL ARCHITECTURE DEAL INTELLIGENCE SUMMARY
")
cat(strrep("═", 80), "
")
print(deal_summary, row.names = FALSE)
cat(strrep("═", 80), "
")

worst_sell    <- deal_summary$Scenario[which.max(
  as.numeric(sub("%", "", deal_summary$SELL_pct))
)]
breach_count  <- sum(deal_summary$Spread_Signal == "BREACH")
impaired_count<- sum(deal_summary$DCF_Signal    == "IMPAIRED")
divergent_count<-sum(deal_summary$BSM_Conv      == "DIVERGENT")

cat("Worst-case (SELL rate):     ", worst_sell, "
")
cat("Spread BREACH scenarios:    ", breach_count, "/", nrow(deal_summary), "
")
cat("DCF IMPAIRED scenarios:     ", impaired_count, "/", nrow(deal_summary), "
")
cat("BSM DIVERGENT scenarios:    ", divergent_count, "/", nrow(deal_summary), "
")
cat(strrep("═", 80), "
")
